In [ ]:
import pandas as pd
import numpy as np

# Zeitphasen einmal zentral definieren – werden in mehreren Notebooks wiederverwendet
# pd.cut mit right=True: Intervalle sind (links, rechts], also z.B. (0, 1899] = bis und mit 1899
PHASE_BINS = [0, 1899, 1949, 1975, 2009, 9999]
PHASE_LABELS = [
    "phase1_fruehphase",
    "phase2_volatile",
    "phase3_konsens",
    "phase4_aufspaltung",
    "phase5_2010_heute",
]

# Hilfsspalte für die Heatmap – wird beim Aggregieren gebraucht, aber nicht exportiert
HEATMAP_META_COLS = ["jahr"]

# Daten einlesen

In [ ]:
# Bereinigten Datensatz laden – Output von 1_data_wrangling.ipynb
df = pd.read_csv("../data/processed/swissvotes_processed.csv")
df.head()

# Uneinigkeit Entdeckt

In [ ]:
# Kontrolle: gibt es Fälle wo BR und BV unterschiedliche Positionen haben?
# Sollte eigentlich nicht vorkommen – interessant wenn doch
mismatch = df["br-pos_label"].notna() & df["bv-pos_label"].notna() & (df["br-pos_label"] != df["bv-pos_label"])
print("Anzahl Abweichungen:", mismatch.sum())
df[mismatch]

## Kongruenzfaktor hinzufügen

In [ ]:
df_with_positions = df.copy()
neue_spalten = {}

# Kongruenzfaktor berechnen: positiv = Akteur und Volk einig, negativ = uneinig
# Formel: (ja-Anteil - 50) / 100 bei Ja-Parole, gespiegelt bei Nein-Parole
# → Werte zwischen -0.5 und +0.5
for col in df:
    if col.startswith('p-') or col.endswith('-pos_label'):
        clean_name = col.replace('_label', '')
        scores = []
        for i, row in df_with_positions.iterrows():
            position = row[col]
            ja_proz = row['volkja-proz']

            # Ja-Parole oder Volksinitiative bevorzugt
            if position in ["Befürwortend", "Vorzug Volksinitiative"]:
                scores.append((ja_proz - 50) / 100)

            # Nein-Parole oder Gegenentwurf bevorzugt
            elif position in ["Ablehnend", "Vorzug Gegenentwurf"]:
                scores.append((50 - ja_proz) / 100)

            # Keine klare Position – kein Score
            elif position in ["Keine Empfehlung", "Stimmfreigabe", "Existiert nicht", "Neutral", "Leer einlegen"]:
                scores.append(np.nan)

            else:
                scores.append(np.nan)

        neue_spalten[f"zustimmung_{clean_name}"] = scores

df_with_positions = pd.concat(
    [df_with_positions, pd.DataFrame(neue_spalten, index=df_with_positions.index)],
    axis=1
)

df_with_positions.head()

In [ ]:
## Zeitliche Abschnitte hinzufügen

In [ ]:
# Zeitphase für jede Abstimmung anhand des Jahres zuweisen
df_with_positions["phase"] = pd.cut(
    df_with_positions["jahr"],
    bins=PHASE_BINS,
    labels=PHASE_LABELS,
)

In [ ]:
# Endresultat speichern – wird in anderen Notebooks als Ausgangsdatensatz geladen
df_with_positions.to_csv('../data/processed/df_with_positions.csv', index=False)
print(df_with_positions)

### Datenset für geografische Heatmap overall

In [ ]:
df_pos = df.copy() # Kopie des Datensatzes damit wir df nicht verändern
PARTEI_HEATMAP_EXCLUDE = {"p-glp_label"}  # GLP nicht mehr in Heatmaps
parteien_cols = [
    c for c in df_pos.columns
    if str(c).startswith("p-") and c not in PARTEI_HEATMAP_EXCLUDE
]
position_cols = ["br-pos_label", "bv-pos_label"] + parteien_cols

canton_cols = [c for c in df.columns if str(c).endswith('-japroz')] # Liste der Kanton-Spalten

rows = [] # Leere Liste für die Zeilen

# Dreistufige Iteration: pro Abstimmung × pro Akteur × pro Kanton
# → für jede Kombination den Kongruenzwert berechnen
for idx, row in df_pos.iterrows(): # 1. Iteration: Zeile (Abstimmung)
    for partei in position_cols: # 2. Iteration: Akteur
        zeile = {
            "partei": partei,
            "jahr": row["jahr"],
        }
        for kanton in canton_cols: # 3. Iteration: Kanton
            ja_proz = row[kanton]
            position = row[partei]
            # Ja-Parole oder Volksinitiative bevorzugt
            if position in ["Befürwortend", "Vorzug Volksinitiative"]:
                zeile[kanton] = (ja_proz - 50)
            # Nein-Parole oder Gegenentwurf bevorzugt
            elif position in ["Ablehnend", "Vorzug Gegenentwurf"]:
               zeile[kanton] = (50 - ja_proz)
            # Neutral
            elif position in ["Keine Empfehlung", "Stimmfreigabe", "Existiert nicht", "Neutral", "Leer einlegen"]:
                zeile[kanton] = np.nan

            else:
                zeile[kanton] = np.nan
        rows.append(zeile)

df_heatmap = pd.DataFrame(rows)
# Jahres-Hilfsspalte weglassen und pro Akteur über alle Kantone mitteln
df_heatmap_with_positions = (
    df_heatmap.drop(columns=HEATMAP_META_COLS)
    .groupby("partei")
    .mean(numeric_only=True)
)
df_heatmap_with_positions.head()

In [ ]:
# Heatmap-Daten (Gesamtperiode) speichern
df_heatmap_with_positions.to_csv('../data/processed/df_heatmap_with_positions.csv', index=True)

### Geografische Heatmap nach Zeitphase (5 Perioden)

In [ ]:
# Gleiche Berechnung, aber nach Zeitphase aufgeteilt
df_heatmap["phase"] = pd.cut(
    df_heatmap["jahr"],
    bins=PHASE_BINS,
    labels=PHASE_LABELS,
)

df_heatmap_by_phase = (
    df_heatmap.drop(columns=HEATMAP_META_COLS)
    .groupby(["partei", "phase"], as_index=False)
    .mean(numeric_only=True)
)
df_heatmap_by_phase.head()

In [ ]:
df_heatmap_by_phase.to_csv('../data/processed/df_heatmap_by_phase.csv', index=True)